**Google Colab setup**

In [2]:
!pip -q install pyspark
from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
uploaded = files.upload() # Browse and select all six CSVs
spark = SparkSession.builder.appName("EnergyAssessment").getOrCreate()
def load_csv(filename):
  return spark.read.csv(filename, header=True, inferSchema=True)
readings_df = load_csv("meter_readings_90000.csv")

Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv


**SECTION 01 / DATA PREPARATION & ANALYSIS**

**Tasks 01-13**

In [3]:
maintenance_tickets_df = spark.read.csv(
    'maintenance_tickets_1600.csv',
    header=True, inferSchema=True)

In [4]:
meter_master_df = spark.read.csv(
    'meter_master.csv',
    header=True, inferSchema=True)

In [5]:
meter_readings_df = spark.read.csv(
    'meter_readings_90000.csv',
    header=True, inferSchema=True)

In [6]:
meter_reading_extra_df = spark.read.csv(
    'meter_readings_extra_12000.csv',
    header=True, inferSchema=True)


In [7]:
substation_df = spark.read.csv(
    'substation_master.csv',
    header=True, inferSchema=True)

In [8]:
tariff_master_df = spark.read.csv(
    'tariff_master.csv',
    header=True, inferSchema=True)

**A. Inspect, select and transform**

In [9]:
#1
meter_readings_df.printSchema()
print("Row Count:",meter_readings_df.count())
meter_readings_df.show(10)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Row Count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04

In [10]:
#2
new_df = meter_readings_df.select(
    'reading_id',
    'meter_id',
    'reading_timestamp',
    'energy_kwh',
    'reading_status'
)
new_df.show()
new_df1 = meter_readings_df.withColumnRenamed(
    'source_type',
    'collection_mode'
)
new_df1.show()

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|         Valid|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|         Valid|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|     Estimated|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|         Valid|
| RD0000005|   M0135|2026-05-27 01:45:00|      4.39|         Valid|
| RD0000006|   M0139|2026-02-20 02:15:00|      0.36|         Valid|
| RD0000007|   M0142|2026-02-01 05:30:00|     79.47|         Valid|
| RD0000008|   M0126|2026-03-01 20:45:00|     28.35|      Rejected|
| RD0000009|   M0005|2026-05-03 10:30:00|      1.96|     Estimated|
| RD0000010|   M0052|2026-04-10 05:45:00|     22.52|         Valid|
| RD0000011|   M0137|2026-02-03 16:00:00|     10.51|         Valid|
| RD0000012|   M0009|2026-05-01 00:00:00|      1

In [11]:
#3
valid_and_20above_df = meter_readings_df.filter(
    (F.col('reading_status') == 'Valid') &
    (F.col('energy_kwh') > 20)
).sort('energy_kwh',ascending=False)
valid_and_20above_df.show()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0041703|   M0101|2026-03-06 22:30:00|     110.0|    213.1|       0.822|             0|         Valid| SmartMeter|                 31.1|      OP010|
| RD0077494|   M0014|2026-05-05 17:30:00|    109.99|     NULL|       0.771|             0|         Valid|     Manual|                 39.0|      OP031|
| RD0034412|   M0026|2026-02-24 21:15:00|    109.96|    231.6|       0.964|             0|         Valid| SmartMeter|                 25.3|      OP017|
| RD0039063|   M0101|2026-02-01 00:45:00|    109.96|    231.6|       0.835|             

In [12]:
#4
meter_readings_df1 = meter_readings_df.withColumn(
    'energy_band',
    F.when(F.col('energy_kwh')<5, 'Low')
    .when((F.col('energy_kwh') >= 5 ) & (F.col('energy_kwh') <= 25), 'Medium')
    .when(F.col('energy_kwh') > 25, 'High')
    .otherwise('Unknown')
)
meter_readings_df1.show()


+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|energy_band|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|     Medium|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|        Low|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|     Medium|
| RD0000004|   M

**B. Null values and data cleaning**

In [13]:
#5
meter_readings_df.agg(
    F.sum(F.when(F.col('energy_kwh').isNull(),1).otherwise(0)).alias('Null_Count of energy_kwh'),
    F.sum(F.when(F.col('voltage_v').isNull(),1).otherwise(0)).alias('Null_Count of voltage_v'),
    F.sum(F.when(F.col('power_factor').isNull(),1).otherwise(0)).alias('Null_Count of power_factor'),
    F.sum(F.when(F.col('operator_id').isNull(),1).otherwise(0)).alias('Null_Count of operator_id')
).show()

+------------------------+-----------------------+--------------------------+-------------------------+
|Null_Count of energy_kwh|Null_Count of voltage_v|Null_Count of power_factor|Null_Count of operator_id|
+------------------------+-----------------------+--------------------------+-------------------------+
|                    1620|                   2070|                      1440|                     5400|
+------------------------+-----------------------+--------------------------+-------------------------+



In [14]:
#6
new_df = meter_readings_df.na.fill(
    {'operator_id':"UNASSIGNED", 'power_factor':0.90}
)
new_df.show()
new_df.agg(
    F.sum(F.when(F.col('power_factor').isNull(),1).otherwise(0)).alias('Null_Count of power_factor'),
    F.sum(F.when(F.col('operator_id').isNull(),1).otherwise(0)).alias('Null_Count of operator_id')
).show()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000001|   M0082|2026-04-01 02:00:00|     22.53|    213.1|       0.939|             0|         Valid| SmartMeter|                 36.2|      OP029|
| RD0000002|   M0020|2026-01-31 06:00:00|      2.97|    227.4|       0.845|             0|         Valid| SmartMeter|                 41.2|      OP025|
| RD0000003|   M0054|2026-04-11 10:15:00|      8.61|    239.3|       0.879|             0|     Estimated| SmartMeter|                 20.9|      OP007|
| RD0000004|   M0111|2026-03-14 00:00:00|      1.87|    246.5|       0.831|             

In [15]:
#7
total_rows = meter_readings_df.count()
meter_readings = meter_readings_df.na.drop(
    subset=['energy_kwh','voltage_v']
)
current_rows = meter_readings.count()
print("Number of rows removed = ",total_rows - current_rows)

Number of rows removed =  3651


**C. Joins, groupBy and aggregations**

In [16]:
#8
join_readings_master = meter_readings_df.join(
    meter_master_df,
    on = 'meter_id',
    how = 'inner'
)
join_readings_master.filter(
    F.col('reading_status') == 'Valid'
).groupBy('site_type').agg(
    F.sum(F.col('energy_kwh')).alias('Total Energy')
).show()

+---------------+-----------------+
|      site_type|     Total Energy|
+---------------+-----------------+
|Public Facility|        138870.41|
|    Residential|100863.2900000004|
|     Industrial|710317.4100000021|
|     Commercial|451525.1799999996|
+---------------+-----------------+



In [17]:
#9
join_readings_master_substation_df = join_readings_master.join(
    substation_df,
    on = 'substation_id',
    how = 'inner'
)
join_readings_master_substation_df.filter(
    F.col('reading_status') == 'Valid'
).groupBy('region','city').agg(
    F.sum(F.col('energy_kwh')).alias('Total Energy')
).show()


+-------+-------------+------------------+
| region|         city|      Total Energy|
+-------+-------------+------------------+
|Central|       Indore|176503.91000000012|
|   West|       Mumbai| 52116.20999999999|
|  South|    Hyderabad|51937.789999999884|
|  South|Visakhapatnam|205015.82999999932|
|   West|    Ahmedabad| 138627.7400000004|
|  North|      Lucknow| 142610.3999999998|
|  South|    Bengaluru| 73787.59999999998|
|  North|       Jaipur| 47863.15999999996|
|  South|        Kochi| 204272.4600000002|
|  North|        Delhi| 94234.89000000014|
|  South|      Chennai| 51829.05999999996|
|   West|         Pune|162777.23999999996|
+-------+-------------+------------------+



In [18]:
#10
join_readings_master_substation_traiff_df = join_readings_master_substation_df.join(
    tariff_master_df,
    on = 'tariff_id',
    how = 'inner'
)
join_readings_master_substation_traiff_df.filter(
    (F.col('reading_status')!='Rejected') &
    (F.col('energy_kwh').isNotNull())
).withColumn(
    'energy_charge_inr',
    F.col('energy_kwh') * F.col('rate_per_kwh_inr')
).show()

+-------------------+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+---------------+-----------------+-----------------+------------+-------------+-------+-----------+-----------------+--------------------+---------------+----------------+------------------------+------------------+
|          tariff_id|substation_id|meter_id|reading_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|      site_type|installation_date|rated_capacity_kw|meter_status|         city| region|capacity_mw|commissioned_year|         tariff_name|  site_category|rate_per_kwh_inr|monthly_fixed_charge_inr| energy_charge_inr|
+-------------------+-------------+--------+----------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+-------------

In [19]:
#11
join_maintenance_tickets_meter_master_df = maintenance_tickets_df.join(
    meter_master_df,
    on = 'meter_id',
    how = 'inner'
)
join_maintenance_tickets_meter_master_df.groupBy('site_type','severity').agg(
    F.count('ticket_id').alias('Total Tickets'),
    F.avg('maintenance_cost_inr').alias('Average Maintenance Cost')
).show()


+---------------+--------+-------------+------------------------+
|      site_type|severity|Total Tickets|Average Maintenance Cost|
+---------------+--------+-------------+------------------------+
|    Residential|Critical|           41|      11875.782702702705|
|     Industrial|  Medium|          106|      12043.898526315788|
|     Commercial|Critical|           37|      13424.726216216219|
|     Commercial|     Low|          166|      12223.517692307689|
|Public Facility|    High|           48|      12905.279347826086|
|    Residential|     Low|          194|       12531.90201117318|
|Public Facility|  Medium|           97|      13050.628421052634|
|     Industrial|Critical|           17|      12296.198000000002|
|Public Facility|Critical|           15|       8986.843571428572|
|     Industrial|    High|           63|      12838.869344262299|
|    Residential|    High|          156|      11047.967122302161|
|Public Facility|     Low|           60|      11195.929464285715|
|     Comm

In [20]:
#12
join_readings_master_substation_df.groupBy('substation_id').agg(
    F.avg('voltage_v').alias('Average Voltage'),
    F.sum('outage_minutes').alias('Total Outage Minutes')
).orderBy('Total Outage Minutes',ascending=False).show(5)

+-------------+------------------+--------------------+
|substation_id|   Average Voltage|Total Outage Minutes|
+-------------+------------------+--------------------+
|        SUB10|230.04010567581344|               91224|
|        SUB09|230.04586272640606|               81734|
|        SUB04| 230.0558687466385|               75872|
|        SUB08|229.88012557077673|               75581|
|        SUB03|230.03094896227708|               61662|
+-------------+------------------+--------------------+
only showing top 5 rows


In [21]:
#13
meter_readings_df.groupBy('source_type').agg(
    F.sum(F.when((F.col('reading_status')=='Valid'),1).otherwise(0)).alias('Valid Readings'),
    F.sum(F.when((F.col('reading_status')=='Estimated'),1).otherwise(0)).alias('Estimated Readings'),
    F.sum(F.when((F.col('reading_status')=='Rejected'),1).otherwise(0)).alias('Rejected Readings'),
    F.count('reading_id').alias('Total Readings')
).sort('Total Readings',ascending = False).show()

+-----------+--------------+------------------+-----------------+--------------+
|source_type|Valid Readings|Estimated Readings|Rejected Readings|Total Readings|
+-----------+--------------+------------------+-----------------+--------------+
| SmartMeter|         65521|             10443|             3202|         79166|
|     Manual|          8965|              1405|              464|         10834|
+-----------+--------------+------------------+-----------------+--------------+



**Tasks 14-25**

**D. Date and window functions**

In [22]:
#14
from pyspark.sql.functions import year, month, to_timestamp
join_readings_master_substation_df = join_readings_master_substation_df.withColumn(
    'reading_timestamp',
    to_timestamp('reading_timestamp')
)
join_readings_master_substation_df = join_readings_master_substation_df.withColumn(
    'year',
    year('reading_timestamp')
)
join_readings_master_substation_df = join_readings_master_substation_df.withColumn(
    'month',
    month('reading_timestamp')
)
join_readings_master_substation_df.filter(
    F.col('reading_status') == 'Valid'
).groupBy('region','year','month').agg(
    F.sum(F.col('energy_kwh')).alias('Total Energy')
).orderBy('region').show()

+-------+----+-----+------------------+
| region|year|month|      Total Energy|
+-------+----+-----+------------------+
|Central|2026|    5|30048.879999999986|
|Central|2026|    6| 27536.84999999999|
|Central|2026|    2|28161.410000000007|
|Central|2026|    3|          29805.85|
|Central|2026|    4|29475.339999999982|
|Central|2026|    1| 31475.57999999999|
|  North|2026|    3| 51094.61000000006|
|  North|2026|    2|45444.249999999985|
|  North|2026|    6| 47072.32999999992|
|  North|2026|    1|48849.309999999925|
|  North|2026|    5| 48796.93000000004|
|  North|2026|    4| 43451.02000000005|
|  South|2026|    1| 97547.38999999987|
|  South|2026|    2| 90864.80000000006|
|  South|2026|    5| 101204.8699999999|
|  South|2026|    4| 97266.94000000005|
|  South|2026|    3|102258.77000000005|
|  South|2026|    6| 97699.97000000002|
|   West|2026|    1| 59967.56999999997|
|   West|2026|    4| 57591.23999999994|
+-------+----+-----+------------------+
only showing top 20 rows


In [23]:
#15
substation_energy_df = join_readings_master_substation_df.filter(
    F.col('reading_status') == 'Valid'
).groupBy('substation_id','meter_id').agg(
    F.sum('energy_kwh').alias('total_valid_energy')
)

window_spec = Window.partitionBy('substation_id').orderBy(F.col('total_valid_energy').desc())

new_df = substation_energy_df.withColumn(
    'dense_rank',
    F.dense_rank().over(window_spec)
)
new_df.filter(
    F.col('dense_rank') <= 3
).show()


+-------------+--------+------------------+----------+
|substation_id|meter_id|total_valid_energy|dense_rank|
+-------------+--------+------------------+----------+
|        SUB01|   M0130|           9364.98|         1|
|        SUB01|   M0110| 9259.750000000002|         2|
|        SUB01|   M0068| 8832.580000000005|         3|
|        SUB02|   M0071|          29242.04|         1|
|        SUB02|   M0145|28641.319999999992|         2|
|        SUB02|   M0133| 7060.140000000003|         3|
|        SUB03|   M0096| 9632.740000000003|         1|
|        SUB03|   M0080| 9525.940000000002|         2|
|        SUB03|   M0073| 9369.789999999995|         3|
|        SUB04|   M0038|          29505.88|         1|
|        SUB04|   M0099|29216.319999999996|         2|
|        SUB04|   M0098|28613.460000000014|         3|
|        SUB05|   M0094| 9659.639999999996|         1|
|        SUB05|   M0050| 9570.859999999997|         2|
|        SUB05|   M0008| 9524.780000000004|         3|
|        S

In [24]:
#16
valid_readings_df = meter_readings_df.filter(F.col('reading_status') == 'Valid').withColumn(
    'reading_timestamp',
    F.to_timestamp('reading_timestamp')
)


window_spec = Window.partitionBy('meter_id').orderBy('reading_timestamp', 'reading_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)

new_df = valid_readings_df.withColumn(
    'running_total',
    F.sum('energy_kwh').over(window_spec)
)

new_df.select('reading_id', 'meter_id', 'reading_timestamp', 'energy_kwh', 'running_total').show(20)

+----------+--------+-------------------+----------+------------------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|     running_total|
+----------+--------+-------------------+----------+------------------+
| RD0083587|   M0001|2026-01-01 01:15:00|     22.27|             22.27|
| RD0086275|   M0001|2026-01-01 03:30:00|      4.05|             26.32|
| RD0088204|   M0001|2026-01-01 09:45:00|     11.28|              37.6|
| RD0005149|   M0001|2026-01-02 04:45:00|     11.76|             49.36|
| RD0035530|   M0001|2026-01-02 08:15:00|     10.06|             59.42|
| RD0070104|   M0001|2026-01-02 18:30:00|     15.83|             75.25|
| RD0006677|   M0001|2026-01-03 00:30:00|     17.06|             92.31|
| RD0021042|   M0001|2026-01-03 06:00:00|     17.14|            109.45|
| RD0032205|   M0001|2026-01-03 08:30:00|     20.46|            129.91|
| RD0074606|   M0001|2026-01-03 18:15:00|      20.4|            150.31|
| RD0019547|   M0001|2026-01-04 19:30:00|      6.84|            

**E. Union and deduplication**

In [25]:
#17
print("Before union count:", meter_readings_df.count())
all_union_df = meter_readings_df.unionByName(meter_reading_extra_df, allowMissingColumns=True)
print("After union count:",all_union_df.count())

Before union count: 90000
After union count: 102000


In [26]:
#18
all_union_df.groupBy('reading_id').agg(
    F.count('reading_id').alias('count')
).filter(
    F.col('count') > 1
).select('reading_id').show()
print("Reading count:",all_union_df.select('reading_id').count())
all_union_df1 = all_union_df.dropDuplicates(['reading_id'])
print("Final unique reading count", all_union_df1.select('reading_id').distinct().count())

+----------+
|reading_id|
+----------+
| RD0025848|
| RD0045533|
| RD0064921|
| RD0056575|
| RD0058815|
| RD0022131|
| RD0029598|
| RD0045710|
| RD0030877|
| RD0002679|
| RD0011169|
| RD0028736|
| RD0030669|
| RD0061933|
| RD0015904|
| RD0008188|
| RD0037899|
| RD0000004|
| RD0025628|
| RD0048911|
+----------+
only showing top 20 rows
Reading count: 102000
Final unique reading count 101300


**F. Lazy execution, caching and persistence**

In [27]:
#19
valid_readings = meter_readings_df.filter(
    F.col('reading_status') == 'Valid'
)
add_column = valid_readings.withColumn(
    'energy_band',
    F.when(F.col('energy_kwh')<5, 'Low')
    .when((F.col('energy_kwh') >= 5 ) & (F.col('energy_kwh') <= 25), 'Medium')
    .when(F.col('energy_kwh') > 25, 'High')
    .otherwise('Unknown')
)
grouping = add_column.groupBy('energy_band').agg(
    F.sum('energy_kwh').alias('Total Energy')
)
ordering = grouping.orderBy(
    F.col('Total Energy').desc()
)
#inspect
ordering.explain()

#action
ordering.show()



== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [Total Energy#1160 DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(Total Energy#1160 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2523]
      +- HashAggregate(keys=[energy_band#1159], functions=[sum(energy_kwh#97)])
         +- Exchange hashpartitioning(energy_band#1159, 200), ENSURE_REQUIREMENTS, [plan_id=2520]
            +- HashAggregate(keys=[energy_band#1159], functions=[partial_sum(energy_kwh#97)])
               +- Project [energy_kwh#97, CASE WHEN (energy_kwh#97 < 5.0) THEN Low WHEN ((energy_kwh#97 >= 5.0) AND (energy_kwh#97 <= 25.0)) THEN Medium WHEN (energy_kwh#97 > 25.0) THEN High ELSE Unknown END AS energy_band#1159]
                  +- Filter (isnotnull(reading_status#101) AND (reading_status#101 = Valid))
                     +- FileScan csv [energy_kwh#97,reading_status#101] Batched: false, DataFilters: [isnotnull(reading_status#101), (reading_status#101 = Valid)], Format: CSV, Location: 

In [28]:
#20
import time
valid_readings = meter_readings_df.filter(
    F.col('reading_status') == 'Valid'
)
valid_readings.cache()
start_time = time.time()
valid_readings.count()
end_time = time.time()
print("Execution time count 1:",end_time - start_time)
start_time = time.time()
valid_readings.count()
end_time = time.time()
print("Execution time count 2:",end_time - start_time)
print("Checking cached:", valid_readings.is_cached)
valid_readings.unpersist()
print("Checking uncached:", valid_readings.is_cached)



Execution time count 1: 2.9103286266326904
Execution time count 2: 0.11972546577453613
Checking cached: True
Checking uncached: False


In [29]:
#21
valid_outage_minute = meter_readings_df.filter(
    F.col('outage_minutes') > 0
)
valid_outage_minute.persist(StorageLevel.MEMORY_AND_DISK)

#action 1
valid_outage_minute.count()

#action 2
valid_outage_minute.show()

#release the data
valid_outage_minute.unpersist()

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|  reading_timestamp|energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
| RD0000050|   M0063|2026-01-12 18:30:00|     18.53|    241.5|       0.908|            11|         Valid| SmartMeter|                 32.8|       NULL|
| RD0000055|   M0031|2026-05-04 21:30:00|     15.21|    226.2|       0.895|           108|         Valid| SmartMeter|                 38.9|      OP037|
| RD0000085|   M0069|2026-02-22 19:15:00|     27.29|    221.1|       0.919|           172|         Valid| SmartMeter|                 31.1|      OP031|
| RD0000090|   M0121|2026-03-13 00:45:00|     35.94|    217.1|       0.955|            9

DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [30]:
#22
valid_readings = meter_readings_df.filter(
    F.col('reading_status') == 'Valid'
)

start_time = time.time()
#aggreagtion
valid_readings.groupBy('source_type').agg(
    F.sum(F.col('energy_kwh')).alias('Total Energy')
).show()
end_time = time.time()
print("Execution of aggregation without caching:",end_time - start_time)

valid_readings.cache()
valid_readings.count()
start_time = time.time()
#aggreagtion
valid_readings.groupBy('source_type').agg(
    F.sum(F.col('energy_kwh')).alias('Total Energy')
).show()
end_time = time.time()
print("Execution of aggregation with caching:",end_time - start_time)
valid_readings.unpersist()


+-----------+-----------------+
|source_type|     Total Energy|
+-----------+-----------------+
|     Manual|166859.9499999999|
| SmartMeter|1234716.339999992|
+-----------+-----------------+

Execution of aggregation without caching: 1.7074153423309326
+-----------+-----------------+
|source_type|     Total Energy|
+-----------+-----------------+
|     Manual|166859.9499999999|
| SmartMeter|1234716.339999992|
+-----------+-----------------+

Execution of aggregation with caching: 0.3660404682159424


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

**G. Partitions and output**

In [31]:
from pyspark.sql.functions.builtin import spark_partition_id
#23
repartition_df8 = meter_readings_df.repartition(8)
repartition_df8.groupBy(spark_partition_id()).agg(
    F.count('reading_id').alias('Count')
).show()


+--------------------+-----+
|SPARK_PARTITION_ID()|Count|
+--------------------+-----+
|                   0|11249|
|                   1|11249|
|                   2|11251|
|                   3|11251|
|                   4|11250|
|                   5|11250|
|                   6|11250|
|                   7|11250|
+--------------------+-----+



In [32]:
#24
repartition_df12 = meter_readings_df.repartition(12)
repartition_df3_coalesce = repartition_df12.coalesce(3)
repartition_df3_repartition = repartition_df12.repartition(3)
repartition_df3_coalesce.explain()
repartition_df3_repartition.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Coalesce 3
   +- Exchange RoundRobinPartitioning(12), REPARTITION_BY_NUM, [plan_id=3030]
      +- FileScan csv [reading_id#94,meter_id#95,reading_timestamp#96,energy_kwh#97,voltage_v#98,power_factor#99,outage_minutes#100,reading_status#101,source_type#102,ambient_temperature_c#103,operator_id#104] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/meter_readings_90000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:...


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=3039]
   +- FileScan csv [reading_id#94,meter_id#95,reading_timestamp#96,energy_kwh#97,voltage_v#98,power_factor#99,outage_minutes#100,reading_status#101,source_type#102,ambient_temperature_c#103,operator_id#104] Batched: false, DataFilte

In [33]:
#25
joined = meter_readings.join(
    meter_master_df,
    on = 'meter_id',
    how = 'inner'
)
joined = joined.join(
    substation_df,
    on = 'substation_id',
    how = 'inner'
)
joined.repartition(4).write.mode('overwrite').parquet('output.parquet')

**Final Capstone**

In [35]:
#1
all_readings_df = meter_readings_df.unionByName(
    meter_reading_extra_df,
    allowMissingColumns=True
)
duplicate_readings_df = all_readings_df.groupBy('reading_id').agg(
    F.count('reading_id').alias('count')
).filter(
    F.col('count') > 1
)
duplicate_readings_df.show()
all_readings_df = all_readings_df.dropDuplicates(subset = ['reading_id'])

#2
all_readings_df = all_readings_df.withColumn(
    'billable_energy_kwh',
    F.when(
        (F.col('reading_status') != 'Rejected') &
        (F.col('reading_status').isNotNull()) &
        (F.col('energy_kwh').isNotNull()),
        F.col('energy_kwh')
    ).otherwise(0)
)

all_readings_df = all_readings_df.withColumn(
    'reading_timestamp',
    F.to_timestamp('reading_timestamp')
)


#3
joined = all_readings_df.join(
    meter_master_df,
    on = 'meter_id',
    how = 'inner'
)
joined = joined.join(
    substation_df,
    on = 'substation_id',
    how = 'inner'
)
joined = joined.join(
    tariff_master_df,
    on = 'tariff_id',
    how = 'inner'
)

#4
joined = joined.withColumn(
    'year',
    F.year('reading_timestamp')
).withColumn(
    'month',
    F.month('reading_timestamp')
).withColumn(
    'energy_charge_inr',
    F.col('billable_energy_kwh') *
    F.col('rate_per_kwh_inr')
)

monthly_report_df = joined.groupBy(
    'year',
    'month',
    'region',
    'site_type'
).agg(
    F.sum('billable_energy_kwh').alias('Total Energy'),
    F.sum('energy_charge_inr').alias('Total Charges')
)

monthly_report_df.show()

#5
outage_df = joined.groupBy(
    'year',
    'month',
    'region',
    'site_type'
).agg(
    F.sum(F.coalesce(F.col('outage_minutes'), F.lit(0))).alias('Total Outage Minutes'),
    F.sum(F.when(F.col('outage_minutes') > 0, 1).otherwise(0)).alias('Outage Readings'),
    F.count('*').alias('Total Readings')
)
outage_df = outage_df.withColumn(
    'Outage Percentage',
    F.round(
        F.col('Outage Readings') /
        F.col('Total Readings') * 100,
        2
    )
)

outage_df.show()

#6
group_df = monthly_report_df.join(
    outage_df,
    on=['year', 'month', 'region', 'site_type'],
    how='inner'
)
window_spec = Window.partitionBy('year','month','region').orderBy(
    F.col('Total Charges').desc()
)
new_df = group_df.withColumn(
    'dense_rank',
    F.dense_rank().over(window_spec)
)
new_df.orderBy(
    'year',
    'month',
    'region',
    'dense_rank'
).show()

#7
new_df.cache()

print("Row count:", new_df.count())
new_df.show()

new_df.unpersist()

#8
new_df.repartition(2).write.mode('overwrite').parquet('output.parquet')

+----------+-----+
|reading_id|count|
+----------+-----+
| RD0025848|    2|
| RD0045533|    2|
| RD0064921|    2|
| RD0056575|    2|
| RD0058815|    2|
| RD0022131|    2|
| RD0029598|    2|
| RD0045710|    2|
| RD0030877|    2|
| RD0002679|    2|
| RD0011169|    2|
| RD0028736|    2|
| RD0030669|    2|
| RD0061933|    2|
| RD0015904|    2|
| RD0008188|    2|
| RD0037899|    2|
| RD0000004|    2|
| RD0025628|    2|
| RD0048911|    2|
+----------+-----+
only showing top 20 rows
+----+-----+-------+---------------+------------------+------------------+
|year|month| region|      site_type|      Total Energy|     Total Charges|
+----+-----+-------+---------------+------------------+------------------+
|2026|    8|Central|     Industrial| 9938.689999999993| 90442.07899999994|
|2026|    8|   West|     Industrial|13792.609999999997|        125512.751|
|2026|    7|   West|     Industrial|13524.280000000004|123070.94799999997|
|2026|    3|Central|     Commercial|10829.769999999997| 88804.1139999